In [1]:
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 51.5 MB/s eta 0:00:00


In [2]:
!pip install -q git+https://github.com/facebookresearch/sam2.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 643.0 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.8/155.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.0/128.0 kB 6.5 MB/s eta 0:00:00


In [3]:
 !mkdir -p checkpoints && wget -q -P checkpoints https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt

In [4]:
!ls -la checkpoints

total 180108
drwxr-xr-x 2 root root      4096 Aug 23 06:09 .
drwxr-xr-x 1 root root      4096 Aug 23 06:09 ..
-rw-r--r-- 1 root root 184416285 Sep 29  2024 sam2.1_hiera_small.pt


In [5]:
!mkdir -p /content/road_health_pipeline
!mv /content/config.py /content/build_memory_bank.py /content/sam2_mask.py /content/dinov2_embed.py /content/coreset.py /content/requirements.txt /content/road_health_pipeline/
!ls -la /content/road_health_pipeline

total 36
drwxr-xr-x 2 root root 4096 Aug 23 06:09 .
drwxr-xr-x 1 root root 4096 Aug 23 06:09 ..
-rw-r--r-- 1 root root 4489 Aug 23 06:05 build_memory_bank.py
-rw-r--r-- 1 root root 1756 Aug 23 06:05 config.py
-rw-r--r-- 1 root root 1520 Aug 23 06:05 coreset.py
-rw-r--r-- 1 root root 2988 Aug 23 06:05 dinov2_embed.py
-rw-r--r-- 1 root root  192 Aug 23 06:05 requirements.txt
-rw-r--r-- 1 root root 1631 Aug 23 06:05 sam2_mask.py


In [6]:
!mkdir -p ~/.kaggle && echo KGAT_b0a3471899a173d4f919810b108987f1 > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token

In [7]:
!kaggle datasets download -d ziedkelboussi/rdd2020-dataset

Dataset URL: https://www.kaggle.com/datasets/ziedkelboussi/rdd2020-dataset
License(s): unknown
100% 1.38G/1.38G [01:09<00:00, 21.5MB/s]



In [8]:
!mkdir -p /content/rdd2020_raw
!unzip -q rdd2020-dataset.zip -d /content/rdd2020_raw
!find /content/rdd2020_raw -maxdepth 4 -type d

/content/rdd2020_raw
/content/rdd2020_raw/train
/content/rdd2020_raw/train/Czech
/content/rdd2020_raw/train/Czech/annotations
/content/rdd2020_raw/train/Czech/annotations/xmls
/content/rdd2020_raw/train/Czech/images
/content/rdd2020_raw/train/Japan
/content/rdd2020_raw/train/Japan/annotations
/content/rdd2020_raw/train/Japan/annotations/xmls
/content/rdd2020_raw/train/Japan/images
/content/rdd2020_raw/train/India
/content/rdd2020_raw/train/India/annotations
/content/rdd2020_raw/train/India/annotations/xmls
/content/rdd2020_raw/train/India/images


In [9]:
import xml.etree.ElementTree as ET
import shutil
from pathlib import Path

raw_root = Path("/content/rdd2020_raw/train")
out_dir = Path("/content/road_health_pipeline/data/healthy_roads")
out_dir.mkdir(parents=True, exist_ok=True)

countries = ["Japan", "Czech", "India"]
total_copied = 0
total_checked = 0

for country in countries:
    images_dir = raw_root / country / "images"
    xmls_dir = raw_root / country / "annotations" / "xmls"

    for img_path in images_dir.glob("*.jpg"):
        xml_path = xmls_dir / (img_path.stem + ".xml")
        if not xml_path.exists():
            continue
        total_checked += 1

        tree = ET.parse(xml_path)
        objects = tree.getroot().findall("object")

        if len(objects) == 0:
            # No damage annotated -> healthy road image
            dest = out_dir / f"{country}_{img_path.name}"
            shutil.copy(img_path, dest)
            total_copied += 1

print(f"Checked {total_checked} images total.")
print(f"Copied {total_copied} healthy (no-damage) images to {out_dir}")

Checked 21041 images total.
Copied 6472 healthy (no-damage) images to /content/road_health_pipeline/data/healthy_roads


In [10]:
!ls /content/road_health_pipeline

build_memory_bank.py  coreset.py  dinov2_embed.py   sam2_mask.py
config.py	      data	  requirements.txt


In [11]:
!find /content -iname "config.py" -o -iname "build_memory_bank.py"

/content/road_health_pipeline/build_memory_bank.py
/content/road_health_pipeline/config.py


In [12]:
import re
p = "/content/road_health_pipeline/config.py"
text = open(p).read()
text = text.replace(
    'SAM2_CHECKPOINT = Path("checkpoints/sam2.1_hiera_small.pt")',
    'SAM2_CHECKPOINT = Path("checkpoints/sam2.1_hiera_small.pt")'
)
text = re.sub(r'DEVICE = ".*?"', 'DEVICE = "cuda"', text)
open(p, "w").write(text)
print("Done. DEVICE line now:", [l for l in text.splitlines() if l.startswith("DEVICE")])

Done. DEVICE line now: ['DEVICE = "cuda"   # "cuda" if you have a GPU for this offline step, else "cpu"']


In [13]:
!find /content -iname "sam2.1_hiera_small.pt"

/content/checkpoints/sam2.1_hiera_small.pt


In [14]:
!mkdir -p /content/road_health_pipeline/checkpoints
!mv /content/checkpoints/sam2.1_hiera_small.pt /content/road_health_pipeline/checkpoints/
!ls -la /content/road_health_pipeline/checkpoints

total 180108
drwxr-xr-x 2 root root      4096 Aug 23 06:10 .
drwxr-xr-x 4 root root      4096 Aug 23 06:10 ..
-rw-r--r-- 1 root root 184416285 Sep 29  2024 sam2.1_hiera_small.pt


In [15]:
%cd /content/road_health_pipeline
!python build_memory_bank.py

/content/road_health_pipeline
Found 6472 healthy-road images.
Traceback (most recent call last):
  File "/content/road_health_pipeline/build_memory_bank.py", line 128, in <module>
    main()
    ~~~~^^
  File "/content/road_health_pipeline/build_memory_bank.py", line 52, in main
    masker = RoadMasker(device=config.DEVICE)
  File "/content/road_health_pipeline/sam2_mask.py", line 24, in __init__
    sam2_model = build_sam2(
        config.SAM2_MODEL_CFG,
        str(config.SAM2_CHECKPOINT),
        device=device,
    )
  File "/usr/local/lib/python3.13/dist-packages/sam2/build_sam.py", line 94, in build_sam2
    model = model.to(device)
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 1384, in to
    return self._apply(convert)
           ~~~~~~~~~~~^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 934, in _apply
    module._apply(fn)
    ~~~~~~~~~~~~~^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/n

In [16]:
%%writefile /content/road_health_pipeline/coreset.py
"""
Greedy k-center coreset subsampling (the same idea PatchCore uses).

Why: if you keep every road patch embedding from every healthy-road image, the
memory bank can easily reach millions of vectors -- too big to ship to and search
on an 8GB Pi. Coreset subsampling picks a small, spread-out subset that still
covers the diversity of "normal road" well, by repeatedly picking whichever
remaining point is farthest from everything already picked.

Two implementations:
  - k_center_greedy: plain numpy/CPU. Fine up to ~50k-100k points, but selecting
    20,000 points out of millions this way can take HOURS (it's O(n*k) with a
    full distance sweep every iteration).
  - k_center_greedy_gpu: same algorithm, vectorized on GPU with torch. On a
    Colab T4, this handles millions of points in a couple of minutes instead.

random_presample: for very large point clouds, randomly cut down to a smaller
pool BEFORE running greedy selection. This trades a bit of coverage quality for
speed and is often the pragmatic choice when n is in the millions.
"""

import numpy as np


def random_presample(embeddings: np.ndarray, n_target: int, seed: int = 42) -> np.ndarray:
    """Returns indices of a random subset of size min(n_target, N)."""
    rng = np.random.default_rng(seed)
    n = embeddings.shape[0]
    if n_target >= n:
        return np.arange(n)
    return rng.choice(n, size=n_target, replace=False)


def k_center_greedy(embeddings: np.ndarray, n_select: int, seed: int = 42) -> np.ndarray:
    """
    CPU/numpy version. embeddings: (N, C) float32 array.
    Only use this directly for N up to roughly 50k-100k points -- beyond that,
    use k_center_greedy_gpu or random_presample first.
    """
    rng = np.random.default_rng(seed)
    n = embeddings.shape[0]
    if n_select >= n:
        return np.arange(n)

    selected = [int(rng.integers(0, n))]
    min_dist = np.linalg.norm(embeddings - embeddings[selected[0]], axis=1)

    for _ in range(n_select - 1):
        next_idx = int(np.argmax(min_dist))
        selected.append(next_idx)
        new_dist = np.linalg.norm(embeddings - embeddings[next_idx], axis=1)
        min_dist = np.minimum(min_dist, new_dist)

    return np.array(selected, dtype=np.int64)


def k_center_greedy_gpu(embeddings: np.ndarray, n_select: int, seed: int = 42,
                         device: str = "cuda", batch_report_every: int = 2000) -> np.ndarray:
    """
    GPU-accelerated k-center greedy using torch. Same algorithm as
    k_center_greedy, but each "distance to all points" sweep runs as one
    vectorized GPU op instead of a numpy CPU loop -- this is roughly 50-200x
    faster in practice, turning an hours-long CPU run into minutes.
    """
    import torch

    rng = np.random.default_rng(seed)
    n = embeddings.shape[0]
    if n_select >= n:
        return np.arange(n)

    x = torch.from_numpy(embeddings).to(device=device, dtype=torch.float32)

    first = int(rng.integers(0, n))
    selected = [first]
    min_dist = torch.linalg.norm(x - x[first], dim=1)

    for i in range(n_select - 1):
        next_idx = int(torch.argmax(min_dist).item())
        selected.append(next_idx)
        new_dist = torch.linalg.norm(x - x[next_idx], dim=1)
        min_dist = torch.minimum(min_dist, next_dist if False else new_dist)

        if (i + 1) % batch_report_every == 0:
            print(f"  coreset: selected {i + 1}/{n_select - 1} points...")

    return np.array(selected, dtype=np.int64)

Overwriting /content/road_health_pipeline/coreset.py


In [17]:
%%writefile /content/road_health_pipeline/build_memory_bank.py
"""
STEP 1 (offline, one-time): build the healthy-road memory bank.
"""

import json
import time
from pathlib import Path

import faiss
import numpy as np
from PIL import Image
from tqdm import tqdm

import config
from coreset import k_center_greedy, k_center_greedy_gpu, random_presample
from dinov2_embed import Dinov2Embedder
from sam2_mask import RoadMasker

PRESAMPLE_THRESHOLD = 300_000

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}


def find_images(root: Path):
    return sorted(p for p in root.rglob("*") if p.suffix.lower() in IMG_EXTS)


def load_rgb(path: Path) -> np.ndarray:
    return np.array(Image.open(path).convert("RGB"))


def main():
    t0 = time.time()
    config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    image_paths = find_images(config.HEALTHY_ROADS_DIR)
    if not image_paths:
        raise SystemExit(f"No images found under {config.HEALTHY_ROADS_DIR}")
    print(f"Found {len(image_paths)} healthy-road images.")

    masker = RoadMasker(device=config.DEVICE)
    embedder = Dinov2Embedder(device=config.DEVICE)

    all_embeddings = []
    per_image_counts = {}
    skipped = []

    for path in tqdm(image_paths, desc="Extracting road patch embeddings"):
        try:
            image_rgb = load_rgb(path)
            road_mask = masker.get_road_mask(image_rgb)

            if road_mask.sum() == 0:
                skipped.append(str(path))
                continue

            embeddings, _coords = embedder.extract_road_patch_embeddings(image_rgb, road_mask)
            if len(embeddings) == 0:
                skipped.append(str(path))
                continue

            all_embeddings.append(embeddings)
            per_image_counts[str(path)] = int(len(embeddings))
        except Exception as e:
            print(f"  [warn] failed on {path}: {e}")
            skipped.append(str(path))

    if not all_embeddings:
        raise SystemExit("No embeddings extracted -- check SAM2 masking and ROI settings.")

    all_embeddings = np.concatenate(all_embeddings, axis=0)
    print(f"Extracted {all_embeddings.shape[0]} total road-patch embeddings "
          f"(dim={all_embeddings.shape[1]}) from {len(per_image_counts)} images.")
    if skipped:
        print(f"Skipped {len(skipped)} images (no road mask found or error).")

    raw_checkpoint_path = config.OUTPUT_DIR / "raw_embeddings_checkpoint.npy"
    np.save(raw_checkpoint_path, all_embeddings)
    print(f"Checkpointed raw embeddings to {raw_checkpoint_path} "
          f"({all_embeddings.nbytes / 1e6:.1f} MB) before running coreset.")

    n_select = min(
        config.CORESET_MAX_POINTS,
        max(1, int(all_embeddings.shape[0] * config.CORESET_RATIO)),
    )

    coreset_source = all_embeddings
    if all_embeddings.shape[0] > PRESAMPLE_THRESHOLD:
        print(f"Randomly presampling {all_embeddings.shape[0]} points down to "
              f"{PRESAMPLE_THRESHOLD} before greedy selection (keeps runtime sane)...")
        presample_idx = random_presample(all_embeddings, PRESAMPLE_THRESHOLD, seed=config.SEED)
        coreset_source = all_embeddings[presample_idx]

    print(f"Running coreset subsampling -> keeping {n_select} points "
          f"from a pool of {coreset_source.shape[0]}...")

    if config.DEVICE == "cuda":
        idx = k_center_greedy_gpu(coreset_source, n_select, seed=config.SEED, device="cuda")
    else:
        idx = k_center_greedy(coreset_source, n_select, seed=config.SEED)

    coreset_embeddings = coreset_source[idx].astype(np.float32)

    dim = coreset_embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(coreset_embeddings)

    np.save(config.OUTPUT_DIR / "embeddings.npy", coreset_embeddings)
    faiss.write_index(index, str(config.OUTPUT_DIR / "index.faiss"))

    metadata = {
        "dinov2_model": config.DINOV2_MODEL_NAME,
        "patch_size": config.PATCH_SIZE,
        "dinov2_input_size": config.DINOV2_INPUT_SIZE,
        "roi_box_fractions": config.ROI_BOX_FRACTIONS,
        "embedding_dim": dim,
        "n_source_images": len(per_image_counts),
        "n_skipped_images": len(skipped),
        "n_total_patch_embeddings": int(all_embeddings.shape[0]),
        "n_coreset_points": int(coreset_embeddings.shape[0]),
        "coreset_ratio_config": config.CORESET_RATIO,
        "coreset_max_points_config": config.CORESET_MAX_POINTS,
        "build_time_seconds": round(time.time() - t0, 1),
    }
    with open(config.OUTPUT_DIR / "metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)

    print(f"\nDone in {metadata['build_time_seconds']}s.")
    print(f"Memory bank written to: {config.OUTPUT_DIR.resolve()}")
    print(json.dumps(metadata, indent=2))


if __name__ == "__main__":
    main()

Overwriting /content/road_health_pipeline/build_memory_bank.py


In [21]:
%cd /content/road_health_pipeline
!python build_memory_bank.py

/content/road_health_pipeline
Found 6472 healthy-road images.
Traceback (most recent call last):
  File "/content/road_health_pipeline/build_memory_bank.py", line 134, in <module>
    main()
    ~~~~^^
  File "/content/road_health_pipeline/build_memory_bank.py", line 41, in main
    masker = RoadMasker(device=config.DEVICE)
  File "/content/road_health_pipeline/sam2_mask.py", line 24, in __init__
    sam2_model = build_sam2(
        config.SAM2_MODEL_CFG,
        str(config.SAM2_CHECKPOINT),
        device=device,
    )
  File "/usr/local/lib/python3.13/dist-packages/sam2/build_sam.py", line 94, in build_sam2
    model = model.to(device)
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 1384, in to
    return self._apply(convert)
           ~~~~~~~~~~~^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 934, in _apply
    module._apply(fn)
    ~~~~~~~~~~~~~^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/n

In [20]:
!zip -r /content/memory_bank.zip output/memory_bank

updating: output/memory_bank/ (stored 0%)
